# Project 91 — Local Coding Copilot
## Search One Repo and Answer Code Questions

**Stack:** Ollama · LangChain · ChromaDB · Local Code Retrieval · Jupyter

## Project Overview

This notebook builds a **local coding copilot** that indexes the source files of a code
repository, stores them in a vector database, and uses retrieval-augmented generation (RAG)
to answer natural-language questions about the codebase.

Everything runs **locally** — the LLM runs via Ollama, embeddings are computed locally, and
the vector store is a local ChromaDB instance. No data ever leaves your machine.

### What You Will Learn

1. How to load and chunk source code files with language-aware splitting
2. How to create a code-search vector index using Ollama embeddings + ChromaDB
3. How to build a RAG chain that answers code questions grounded in real source files
4. How to evaluate retrieval quality and answer groundedness qualitatively
5. Practical patterns for navigating unfamiliar codebases with LLM assistance

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` and `nomic-embed-text` pulled |
| **Python packages** | `langchain`, `langchain-ollama`, `langchain-community`, `langchain-text-splitters`, `chromadb` |

```bash
# Pull the models (run in terminal)
ollama pull qwen3:8b
ollama pull nomic-embed-text
```

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community langchain-text-splitters chromadb

## Step 1 — Imports and Configuration

In [ ]:
import os
import textwrap
from pathlib import Path

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Configuration ─────────────────────────────────────────────
OLLAMA_MODEL = "qwen3:8b"
EMBED_MODEL = "nomic-embed-text"
CHROMA_DIR = "sample_data/code_chroma"
COLLECTION = "repo_code"

# File extensions to index and their Language enum for smart splitting
LANG_MAP = {
    ".py": Language.PYTHON,
    ".js": Language.JS,
    ".ts": Language.TS,
    ".java": Language.JAVA,
    ".go": Language.GO,
    ".rs": Language.RUST,
    ".rb": Language.RUBY,
    ".md": Language.MARKDOWN,
}

print("Configuration ready.")

## Step 2 — Initialize LLM and Embeddings

We use **Qwen3 8B** (via Ollama) as the reasoning LLM and **nomic-embed-text** for
creating vector embeddings of code chunks. Both run entirely on your local machine.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.1)
embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# Quick connectivity check
test = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test.content.strip()[:20]}")

test_vec = embeddings.embed_query("hello")
print(f"Embeddings ready — dim={len(test_vec)}")

## Step 3 — Create a Sample Repository

For reproducibility we create a small but realistic sample Python project in memory.
This simulates indexing a real repository. To use your own repo, replace
`REPO_ROOT` with the path to any local code directory (see Step 12).

In [ ]:
# ── Sample repository files ───────────────────────────────────
# A small task-management API project (5 Python files + 1 Markdown)

SAMPLE_REPO = {
    "taskmanager/models.py": textwrap.dedent('''\
        """Data models for the task manager application."""
        from dataclasses import dataclass, field
        from datetime import datetime
        from enum import Enum
        from typing import Optional
        import uuid


        class Priority(Enum):
            LOW = "low"
            MEDIUM = "medium"
            HIGH = "high"
            CRITICAL = "critical"


        class Status(Enum):
            TODO = "todo"
            IN_PROGRESS = "in_progress"
            DONE = "done"
            ARCHIVED = "archived"


        @dataclass
        class Task:
            """Represents a single task in the system."""
            title: str
            description: str = ""
            priority: Priority = Priority.MEDIUM
            status: Status = Status.TODO
            assignee: Optional[str] = None
            tags: list[str] = field(default_factory=list)
            id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
            created_at: datetime = field(default_factory=datetime.utcnow)
            due_date: Optional[datetime] = None

            def is_overdue(self) -> bool:
                """Check whether the task is past its due date."""
                if self.due_date is None:
                    return False
                return datetime.utcnow() > self.due_date and self.status != Status.DONE

            def to_dict(self) -> dict:
                return {
                    "id": self.id,
                    "title": self.title,
                    "description": self.description,
                    "priority": self.priority.value,
                    "status": self.status.value,
                    "assignee": self.assignee,
                    "tags": self.tags,
                    "created_at": self.created_at.isoformat(),
                    "due_date": self.due_date.isoformat() if self.due_date else None,
                }


        @dataclass
        class Project:
            """A project that groups related tasks."""
            name: str
            owner: str
            tasks: list[Task] = field(default_factory=list)
            id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

            def add_task(self, task: Task) -> None:
                self.tasks.append(task)

            def get_tasks_by_status(self, status: Status) -> list[Task]:
                return [t for t in self.tasks if t.status == status]

            def completion_rate(self) -> float:
                if not self.tasks:
                    return 0.0
                done = sum(1 for t in self.tasks if t.status == Status.DONE)
                return done / len(self.tasks)
    '''),

    "taskmanager/storage.py": textwrap.dedent('''\
        """JSON-file storage backend for tasks and projects."""
        import json
        from pathlib import Path
        from datetime import datetime
        from typing import Optional
        from taskmanager.models import Task, Project, Priority, Status


        class JSONStorage:
            """Persist tasks and projects to a local JSON file."""

            def __init__(self, filepath: str = "tasks.json"):
                self.filepath = Path(filepath)
                self._data: dict = {"projects": {}, "tasks": {}}
                if self.filepath.exists():
                    self._load()

            def _load(self) -> None:
                with open(self.filepath, "r") as f:
                    self._data = json.load(f)

            def _save(self) -> None:
                with open(self.filepath, "w") as f:
                    json.dump(self._data, f, indent=2, default=str)

            def save_task(self, task: Task) -> str:
                """Save a task and return its ID."""
                self._data["tasks"][task.id] = task.to_dict()
                self._save()
                return task.id

            def get_task(self, task_id: str) -> Optional[dict]:
                """Retrieve a task by ID. Returns None if not found."""
                return self._data["tasks"].get(task_id)

            def delete_task(self, task_id: str) -> bool:
                """Delete a task. Returns True if found and deleted."""
                if task_id in self._data["tasks"]:
                    del self._data["tasks"][task_id]
                    self._save()
                    return True
                return False

            def list_tasks(self, status: Optional[str] = None) -> list[dict]:
                """List all tasks, optionally filtered by status."""
                tasks = list(self._data["tasks"].values())
                if status:
                    tasks = [t for t in tasks if t["status"] == status]
                return tasks

            def search_tasks(self, query: str) -> list[dict]:
                """Search tasks by title or description substring."""
                query_lower = query.lower()
                return [
                    t for t in self._data["tasks"].values()
                    if query_lower in t["title"].lower()
                    or query_lower in t["description"].lower()
                ]
    '''),

    "taskmanager/service.py": textwrap.dedent('''\
        """Business logic layer for task management operations."""
        from datetime import datetime, timedelta
        from typing import Optional
        from taskmanager.models import Task, Project, Priority, Status
        from taskmanager.storage import JSONStorage


        class TaskService:
            """High-level operations on tasks."""

            def __init__(self, storage: Optional[JSONStorage] = None):
                self.storage = storage or JSONStorage()

            def create_task(
                self,
                title: str,
                description: str = "",
                priority: str = "medium",
                assignee: Optional[str] = None,
                due_days: Optional[int] = None,
            ) -> Task:
                """Create and persist a new task."""
                due = datetime.utcnow() + timedelta(days=due_days) if due_days else None
                task = Task(
                    title=title,
                    description=description,
                    priority=Priority(priority),
                    assignee=assignee,
                    due_date=due,
                )
                self.storage.save_task(task)
                return task

            def complete_task(self, task_id: str) -> bool:
                """Mark a task as done."""
                data = self.storage.get_task(task_id)
                if not data:
                    return False
                data["status"] = Status.DONE.value
                self.storage._data["tasks"][task_id] = data
                self.storage._save()
                return True

            def get_overdue_tasks(self) -> list[dict]:
                """Return tasks that are past their due date."""
                now = datetime.utcnow()
                overdue = []
                for t in self.storage.list_tasks():
                    if t["due_date"] and t["status"] != "done":
                        due = datetime.fromisoformat(t["due_date"])
                        if now > due:
                            overdue.append(t)
                return overdue

            def get_stats(self) -> dict:
                """Return summary statistics about all tasks."""
                tasks = self.storage.list_tasks()
                total = len(tasks)
                by_status = {}
                by_priority = {}
                for t in tasks:
                    by_status[t["status"]] = by_status.get(t["status"], 0) + 1
                    by_priority[t["priority"]] = by_priority.get(t["priority"], 0) + 1
                return {
                    "total": total,
                    "by_status": by_status,
                    "by_priority": by_priority,
                    "overdue": len(self.get_overdue_tasks()),
                }
    '''),

    "taskmanager/cli.py": textwrap.dedent('''\
        """Command-line interface for the task manager."""
        import argparse
        import sys
        from taskmanager.service import TaskService


        def build_parser() -> argparse.ArgumentParser:
            parser = argparse.ArgumentParser(description="Task Manager CLI")
            sub = parser.add_subparsers(dest="command")

            # add
            add_p = sub.add_parser("add", help="Create a new task")
            add_p.add_argument("title")
            add_p.add_argument("--desc", default="")
            add_p.add_argument("--priority", default="medium",
                               choices=["low", "medium", "high", "critical"])
            add_p.add_argument("--assignee", default=None)
            add_p.add_argument("--due-days", type=int, default=None)

            # list
            list_p = sub.add_parser("list", help="List tasks")
            list_p.add_argument("--status", default=None)

            # done
            done_p = sub.add_parser("done", help="Mark task as done")
            done_p.add_argument("task_id")

            # search
            search_p = sub.add_parser("search", help="Search tasks")
            search_p.add_argument("query")

            # stats
            sub.add_parser("stats", help="Show statistics")

            return parser


        def main() -> None:
            parser = build_parser()
            args = parser.parse_args()
            svc = TaskService()

            if args.command == "add":
                task = svc.create_task(
                    title=args.title,
                    description=args.desc,
                    priority=args.priority,
                    assignee=args.assignee,
                    due_days=args.due_days,
                )
                print(f"Created task {task.id}: {task.title}")

            elif args.command == "list":
                for t in svc.storage.list_tasks(args.status):
                    print(f"[{t['status']}] {t['id']} — {t['title']}")

            elif args.command == "done":
                ok = svc.complete_task(args.task_id)
                print("Done!" if ok else "Task not found.")

            elif args.command == "search":
                for t in svc.storage.search_tasks(args.query):
                    print(f"{t['id']} — {t['title']}")

            elif args.command == "stats":
                stats = svc.get_stats()
                print(f"Total: {stats['total']}")
                print(f"By status: {stats['by_status']}")
                print(f"By priority: {stats['by_priority']}")
                print(f"Overdue: {stats['overdue']}")
            else:
                parser.print_help()


        if __name__ == "__main__":
            main()
    '''),

    "taskmanager/utils.py": textwrap.dedent('''\
        """Utility helpers for the task manager."""
        from datetime import datetime


        def format_date(dt: datetime) -> str:
            """Format a datetime for display."""
            return dt.strftime("%Y-%m-%d %H:%M")


        def days_until(dt: datetime) -> int:
            """Return the number of days from now until *dt*."""
            delta = dt - datetime.utcnow()
            return delta.days


        def truncate(text: str, max_len: int = 80) -> str:
            """Truncate a string and add ellipsis if needed."""
            if len(text) <= max_len:
                return text
            return text[:max_len - 3] + "..."


        def slugify(text: str) -> str:
            """Convert text into a URL-safe slug."""
            return (
                text.lower()
                .replace(" ", "-")
                .replace("_", "-")
                .strip("-")
            )
    '''),

    "README.md": textwrap.dedent('''\
        # Task Manager

        A lightweight CLI task manager written in Python.

        ## Quick Start

        ```bash
        python -m taskmanager.cli add "Fix login bug" --priority high --assignee alice
        python -m taskmanager.cli list
        python -m taskmanager.cli done <task_id>
        python -m taskmanager.cli stats
        ```

        ## Architecture

        | Module | Responsibility |
        |---|---|
        | `models.py` | Data classes — Task, Project, Priority, Status |
        | `storage.py` | JSON file persistence |
        | `service.py` | Business logic (create, complete, stats) |
        | `cli.py` | Argument parsing and user interaction |
        | `utils.py` | Date formatting, string helpers |

        ## Data Storage

        Tasks are stored in `tasks.json` in the current working directory.
        The file is created automatically on the first write.
    '''),
}

print(f"Sample repo has {len(SAMPLE_REPO)} files:")
for path in sorted(SAMPLE_REPO):
    lines = SAMPLE_REPO[path].count('\n')
    print(f"  {path} ({lines} lines)")

## Step 4 — Load and Chunk Repository Files

We use LangChain's **language-aware** `RecursiveCharacterTextSplitter` so that chunks
respect function and class boundaries whenever possible. Each chunk keeps metadata about
its source file path, making retrieved results easy to trace.

In [ ]:
def load_repo_documents(repo_files: dict[str, str]) -> list[Document]:
    """Convert a dict of {filepath: content} into LangChain Documents."""
    docs = []
    for filepath, content in repo_files.items():
        ext = Path(filepath).suffix
        docs.append(Document(
            page_content=content,
            metadata={"source": filepath, "extension": ext},
        ))
    return docs


def chunk_code_documents(
    documents: list[Document],
    chunk_size: int = 600,
    chunk_overlap: int = 80,
) -> list[Document]:
    """Split documents using language-aware splitters where possible."""
    all_chunks: list[Document] = []
    for doc in documents:
        ext = doc.metadata.get("extension", "")
        lang = LANG_MAP.get(ext)
        if lang is not None:
            splitter = RecursiveCharacterTextSplitter.from_language(
                language=lang,
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
        else:
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
            )
        chunks = splitter.split_documents([doc])
        all_chunks.extend(chunks)
    return all_chunks


raw_docs = load_repo_documents(SAMPLE_REPO)
chunks = chunk_code_documents(raw_docs)
print(f"Loaded {len(raw_docs)} files -> {len(chunks)} chunks")

# Show a sample chunk
print(f"\n-- Sample chunk (from {chunks[0].metadata['source']}) --")
print(chunks[0].page_content[:300])

## Step 5 — Build the Code Vector Index

We embed every chunk with **nomic-embed-text** and store the vectors in a local
ChromaDB collection. This is the "search engine" that the copilot queries when
answering your questions.

In [ ]:
# Clean previous index if it exists (idempotent)
import shutil
if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name=COLLECTION,
)

print(f"Vector index built — {vectorstore._collection.count()} chunks indexed")

## Step 6 — Code Search (Retrieval Only)

Before wiring up the full RAG chain, let's verify that **retrieval alone** returns
relevant code chunks. This is the foundation — if retrieval is poor, the LLM's
answers will be poor too.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

search_queries = [
    "How is a task created?",
    "Where is the JSON data stored and loaded?",
    "What CLI commands are available?",
]

for query in search_queries:
    print(f"\nQuery: {query}")
    results = retriever.invoke(query)
    for i, doc in enumerate(results, 1):
        src = doc.metadata["source"]
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{i}] {src} — {snippet}...")
    print()

## Step 7 — Build the RAG Coding Copilot Chain

The copilot chain combines:
1. **Retriever** — finds the most relevant code chunks
2. **Prompt** — instructs the LLM to answer grounded in retrieved source code
3. **LLM** — generates a natural-language answer with code references

The system prompt tells the model to cite file paths and include relevant code
snippets in its answer.

In [ ]:
SYSTEM_PROMPT = """You are a local coding copilot. You answer questions about a
codebase using ONLY the retrieved source code provided below.

Rules:
- Always cite the source file path when referencing code.
- Include short code snippets from the retrieved context when helpful.
- If the answer is not in the retrieved context, say so honestly.
- Keep answers concise and developer-friendly.

Retrieved source code:
{context}"""

copilot_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])


def format_docs(docs: list[Document]) -> str:
    """Format retrieved documents into a single context string."""
    parts = []
    for doc in docs:
        src = doc.metadata["source"]
        parts.append(f"--- {src} ---\n{doc.page_content}")
    return "\n\n".join(parts)


copilot_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | copilot_prompt
    | llm
    | StrOutputParser()
)

print("Coding copilot RAG chain ready.")

## Step 8 — Ask Code Questions

Let's test the copilot with a range of questions — from high-level architecture
to specific implementation details.

In [ ]:
questions = [
    "What is the overall architecture of this project? Which modules exist and what does each do?",
    "How does the Task data model work? What fields does it have?",
    "How are tasks persisted to disk? What storage format is used?",
    "How can I mark a task as done using the CLI?",
    "What happens when I search for tasks? Which fields are searched?",
    "How is the completion rate of a project calculated?",
]

for q in questions:
    print(f"\n{'='*70}")
    print(f"Question: {q}")
    print(f"{'='*70}")
    answer = copilot_chain.invoke(q)
    print(answer)
    print()

## Step 9 — Code Explanation Mode

Sometimes you want to point the copilot at a specific file or function and ask it
to **explain** the code. We build a lightweight explain chain that retrieves the
relevant chunks and produces a structured explanation.

In [ ]:
EXPLAIN_PROMPT = """You are a code explainer. Given the source code below, provide:
1. A one-sentence summary of what it does
2. A step-by-step walkthrough of the logic
3. Any edge cases or potential issues

Source code:
{context}"""

explain_prompt = ChatPromptTemplate.from_messages([
    ("system", EXPLAIN_PROMPT),
    ("human", "Explain: {question}"),
])

explain_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | explain_prompt
    | llm
    | StrOutputParser()
)

explain_queries = [
    "the is_overdue method in models.py",
    "the JSONStorage class and how it handles persistence",
    "the get_stats method in service.py",
]

for eq in explain_queries:
    print(f"\n{'='*70}")
    print(f"Explain: {eq}")
    print(f"{'='*70}")
    explanation = explain_chain.invoke(eq)
    print(explanation)
    print()

## Step 10 — Code Review Mode

The copilot can also perform lightweight **code review** — searching for relevant
code patterns and flagging potential issues.

In [ ]:
REVIEW_PROMPT = """You are a senior code reviewer. Based on the retrieved source code,
review for:
1. Bugs or logic errors
2. Missing error handling
3. Security concerns
4. Possible improvements

Be specific — cite file paths and line-level details.

Source code:
{context}"""

review_prompt = ChatPromptTemplate.from_messages([
    ("system", REVIEW_PROMPT),
    ("human", "Review: {question}"),
])

review_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | review_prompt
    | llm
    | StrOutputParser()
)

review_answer = review_chain.invoke("the storage module — focus on data safety and error handling")
print("CODE REVIEW — storage.py")
print("=" * 50)
print(review_answer)

## Step 11 — Retrieval Quality Evaluation

We evaluate retrieval quality by checking whether the **expected source file** appears
in the top-k retrieved chunks. This is a simple but effective proxy for retrieval
precision.

In [ ]:
eval_cases = [
    {"query": "How is a task created?", "expected_source": "taskmanager/service.py"},
    {"query": "What fields does a Task have?", "expected_source": "taskmanager/models.py"},
    {"query": "How is data saved to disk?", "expected_source": "taskmanager/storage.py"},
    {"query": "What CLI commands exist?", "expected_source": "taskmanager/cli.py"},
    {"query": "How to format a date?", "expected_source": "taskmanager/utils.py"},
    {"query": "Project architecture overview", "expected_source": "README.md"},
]

hits = 0
for case in eval_cases:
    results = retriever.invoke(case["query"])
    sources = [doc.metadata["source"] for doc in results]
    found = case["expected_source"] in sources
    hits += int(found)
    status = "PASS" if found else "MISS"
    print(f"{status} '{case['query']}' -> expected {case['expected_source']}")
    if not found:
        print(f"   Got: {sources}")

print(f"\nRetrieval hit rate: {hits}/{len(eval_cases)} ({hits/len(eval_cases)*100:.0f}%)")

## Step 12 — Using Your Own Repository

To point the copilot at a **real** local repository, replace the sample-repo
dictionary with files loaded from disk. Here is a helper function that scans
a directory for supported source files.

> **Note:** This cell is a template — set `REPO_ROOT` to your own repo path to try it.

In [ ]:
def load_repo_from_disk(
    repo_root: str,
    extensions: tuple[str, ...] = (".py", ".js", ".ts", ".md"),
    max_file_size_kb: int = 100,
    max_files: int = 200,
) -> dict[str, str]:
    """Walk a repo directory and return {relative_path: content}.

    Skips hidden dirs, __pycache__, node_modules, .git, and binary files.
    """
    skip_dirs = {".git", "__pycache__", "node_modules", ".venv", "venv", ".tox", "dist", "build"}
    root = Path(repo_root)
    files: dict[str, str] = {}
    count = 0
    for dirpath, dirnames, filenames in os.walk(root):
        # Prune skipped directories
        dirnames[:] = [d for d in dirnames if d not in skip_dirs and not d.startswith(".")]
        for fname in filenames:
            if count >= max_files:
                break
            fpath = Path(dirpath) / fname
            if fpath.suffix not in extensions:
                continue
            if fpath.stat().st_size > max_file_size_kb * 1024:
                continue
            try:
                content = fpath.read_text(encoding="utf-8", errors="ignore")
                rel = str(fpath.relative_to(root)).replace("\\", "/")
                files[rel] = content
                count += 1
            except Exception:
                pass
    return files


# ── Uncomment and set your repo path to try it ──
# REPO_ROOT = "/path/to/your/repo"
# my_repo = load_repo_from_disk(REPO_ROOT)
# my_docs = load_repo_documents(my_repo)
# my_chunks = chunk_code_documents(my_docs)
# if os.path.exists("sample_data/my_repo_chroma"):
#     shutil.rmtree("sample_data/my_repo_chroma")
# vectorstore = Chroma.from_documents(my_chunks, embeddings,
#     persist_directory="sample_data/my_repo_chroma", collection_name="my_repo")
# retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
# print(f"Indexed {len(my_chunks)} chunks from {len(my_repo)} files")

print("load_repo_from_disk() helper ready — set REPO_ROOT to use with your own repo.")

## Step 13 — Limitations and Failure Cases

Let's see how the copilot behaves when asked something **not in the codebase**.
A good RAG system should admit when it doesn't have the answer.

In [ ]:
out_of_scope = [
    "Does this project have database migration support?",
    "How does the authentication system work?",
    "What tests exist for the storage module?",
]

for q in out_of_scope:
    print(f"\nQ: {q}")
    answer = copilot_chain.invoke(q)
    print(f"A: {answer}")
    print("-" * 50)

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Retrieval quality** | Hit-rate on expected source files (Step 11) |
| **Answer groundedness** | Inspected whether answers cite retrieved code, not hallucinations |
| **Out-of-scope handling** | Verified the copilot admits gaps (Step 13) |
| **Explanation quality** | Checked structured explanations include summary + walkthrough (Step 9) |
| **Review quality** | Checked code review identifies real issues with file-level citations (Step 10) |

### Known Limitations

- **Chunk size trade-off**: Smaller chunks improve retrieval precision but lose
  surrounding context. Larger chunks keep context but may dilute relevance.
- **Cross-file reasoning**: The copilot sees only top-k chunks, so questions that
  require tracing imports across many files may get incomplete answers.
- **No AST awareness**: The splitter uses text heuristics, not a real parser. It can
  occasionally split mid-function.
- **Embedding model**: `nomic-embed-text` is a general-purpose model, not code-specific.
  A code-tuned embedding model could improve retrieval.

## How to Improve This Project

1. **Use a code-specific embedding model** (e.g., Jina Code v2) for better semantic match
2. **Add AST-based splitting** — parse Python with `ast` to split at function/class boundaries
3. **Add re-ranking** — use a cross-encoder reranker after initial retrieval to improve precision
4. **Multi-turn conversation** — add chat memory so follow-up questions have context
5. **Hybrid search** — combine vector similarity with BM25 keyword search
6. **Incremental indexing** — watch the repo for file changes and update the index

## What You Learned

- **Local code retrieval** — indexing source files into a vector store with language-aware chunking
- **RAG for code Q&A** — answering natural-language questions grounded in real source code
- **Code explanation and review** — structured analysis of retrieved code snippets
- **Retrieval evaluation** — measuring hit-rate to validate the search index
- **Groundedness checking** — testing out-of-scope questions to verify honest behavior
- **Extensibility** — a `load_repo_from_disk()` helper to point the copilot at any real repository